In [ ]:
import pandas as pd
from zhipuai import ZhipuAI

# ===================== 配置 =====================
API_KEY = "aec8c161eae440bf9a03fb79803594c0.7uGk8vDg8uOIhPrU"
FILE_PATH = "综合受害者画像_LLM最终版.xlsx"
MODEL = "glm-4-flash"
# ======================================================

client = ZhipuAI(api_key=API_KEY)
df = pd.read_excel(FILE_PATH)

# 只保留 年龄 + 诈骗类型 两列，并清洗空值
df_use = df[["受害者年龄", "诈骗类型"]].copy()
df_use = df_use.dropna()
df_use = df_use[df_use["受害者年龄"] != "未知"]
df_use = df_use[df_use["诈骗类型"] != "未知"]

# 把数据拼成文字给 LLM
data_lines = []
for _, row in df_use.iterrows():
    data_lines.append(f"年龄：{row['受害者年龄']}，诈骗类型：{row['诈骗类型']}")

input_text = "\n".join(data_lines[:500])  # 取前500条避免超长

# ===================== LLM 分析（输出每个年龄段 TOP3 诈骗）=====================
prompt = f"""
你是反诈数据分析专家。
根据下面的案例数据，**按年龄段汇总**，输出：
每个年龄段 **最容易遭受的 3 种诈骗类型**。

要求：
1. 只输出结论，不解释、不啰嗦
2. 格式统一：
【xx岁】最容易遭受：1.xxx 2.xxx 3.xxx
3. 不要重复年龄段

案例数据：
{input_text}
"""

# 调用LLM
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    temperature=0.1
)

# 输出最终结果
print("===== LLM 分析：各年龄段 TOP3 诈骗类型 =====")
print(response.choices[0].message.content)